# My first notebook


## Data read


## Json ingestion


In [0]:
df2=spark.read.format("json").option("header",True).option("inferSchema",True).option("multiline",False).load("/Volumes/workspace/default/json_data")
display(df2)

### Csv data ingestion from volumes

In [0]:
df=spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .load("/Volumes/workspace/default/sales_data")
display(df)


In [0]:
df.printSchema()

## DDL Schema


In [0]:
ddl_schema='''
Item_Identifier String,
Item_Weight Double,
Item_Fat_Content String,
Item_Visibility Double,
Item_Type String,
Item_MRP Double,
Outlet_Identifier String,
Outlet_Establishment_Year Integer,
Outlet_Size String,
Outlet_Location_Type String,
Outlet_Type String,
Item_Outlet_Sales Double
'''

In [0]:
df=spark.read.format("csv").option("header",True).schema(ddl_schema).load("/Volumes/workspace/default/sales_data")
display(df)

## Struct Type Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import * 


In [0]:
my_struct_schema=StructType([
    StructField("Item_Identifier", StringType()),
    StructField("Item_Weight", StringType()),
    StructField("Item_Fat_Content", StringType()),
    StructField("Item_Visibility", DoubleType()),
    StructField("Item_Type", StringType()),
    StructField("Item_MRP", DoubleType()),
    StructField("Outlet_Identifier", StringType()),
    StructField("Outlet_Establishment_Year", IntegerType()),
    StructField("Outlet_Size", StringType()),
    StructField("Outlet_Location_Type", StringType()),
    StructField("Outlet_Type", StringType()),
    StructField("Item_Outlet_Sales", DoubleType())
])

In [0]:
df=spark.read.format("csv").option("header",True).schema(my_struct_schema).load("/Volumes/workspace/default/sales_data")
display(df)

In [0]:
df.printSchema()

## Select


In [0]:
df_sel=df.select("Item_Identifier","Item_Weight","Item_Fat_Content")
df_sel.display()


## Using column object 


In [0]:
df_sel_2=df.select(col("Item_Identifier"),col("Item_Weight"),col("Item_Fat_Content"))
df_sel_2.display()

## Alias
### It is used to give temproary names to the columns we can do it using the col object


In [0]:
df_sel_3=df.select(col("Item_Identifier").alias("Item_Id"))
df_sel_3.display()

## Filter/Where
'''
1.Filter data where fat content=Regular
2.Slice the data with item type=Soft drinks and weight<10
3.Fetch the data where with Tier in(Tier 1 or Tier 2) and outlet size is Null'''

### Scenario 1

In [0]:
df_s1=df.filter(col("Item_Fat_Content")=="Regular")
display(df_s1)

## Scenario 2

In [0]:
df.filter((col("Item_Type")=="Soft Drinks") & (col("Item_Weight")<10)).display()

## Scenario 3

In [0]:
df.filter((col("Outlet_Location_Type").isin(["Tier 1","Tier 2"])) & col("Outlet_Size").isNull()).display()


## withColumnName - It is used to change the name of the column at data frame level where as alias is used to just give the name to the tranformed column.

In [0]:
df.withColumnRenamed("Item_Weight","Item_Wt").display()

## withColumn Transformation - We can use this transformation to add a new column or modify the existing column

## Scenario 1 new column

In [0]:
df=df.withColumn("flag",lit("new"))
df.display()


In [0]:
df.withColumn("multiply",col('Item_Weight')*col('Item_MRP')).display()

## Scenario 2 - Modify the existing columns contents based on condition
### We will use regexp_replace()- it is a function used to replace the existing values 

In [0]:
df.withColumn("Item_Fat_Content",regexp_replace(col('Item_Fat_Content'),"Regular","Reg"))\
    .withColumn("Item_Fat_Content",regexp_replace(col('Item_Fat_Content'),"Low Fat","LF"))\
    .display()

## Typecasting-It is used to change the datatype of column from one to other.Used when we perform aggregation or joins.

In [0]:
from pyspark.sql.types import StringType
df = df.withColumn('Item_Weight', col('Item_Weight').cast(StringType()))

In [0]:
df.printSchema()

## Sort or Order by--We use this to sort our data based on any col asc/desc

### Scenario 1- Sort the data based on item_weight in desc order.

In [0]:
df.sort(col('Item_Weight').desc()).display()

### Scenario 2- Sort on Item_Visibility in asc order.

In [0]:
df.sort(col('Item_Visibility').asc()).display()

### Scenario 3- Sort based on multiple columns Item_Weight and Item_Visibility in desc order.

In [0]:
df.sort(['Item_Weight','Item_Visibility'],ascending=[0,0]).display()

### Limit -It is used to get limited number of records like first 5,10.

In [0]:
df.limit(10).display()

### Drop -It is used to drop col or cols from df 

### Scenario 1- Drop 1 column

In [0]:
df.drop('Item_Visibility').display()


## Scenario 2- Drop mutiple columns

In [0]:
df.drop('Item_Visibility','Item_Type').display()

## Drop duplicates -Used to remove duplicate columns from our dataframe 

In [0]:
df.dropDuplicates().display()

In [0]:
df.dropDuplicates(subset=['Item_Type']).display()

### We can use distinct as well to get the distinct keywords rather than using dropDuplicates me

### Union and union by name are two functions which we use to combine two dataframes.

In [0]:
df2=df.select(col('Item_Fat_Content'),col('Item_MRP'))
df3=df.select(col('Item_Fat_Content'),col('Item_MRP'))
df3=df3.union(df2)
display(df3.count(),'   ',df3.count()/2,'   ',df2.count())

## Preparing dataframes

In [0]:
data1=[(1,'Kid1'),(2,'Kid2')]
data2=[(3,'Kid3'),(4,'Kid4')]
schema=('id String','name String')

df_1=spark.createDataFrame(data=data1,schema=schema)
df_2=spark.createDataFrame(data=data2,schema=schema)
df_3=df_1.union(df_2)
display(df_3)

%
## The need of unionByName function is when we the order of columns of two dataframes can be different and we want to unify them.

In [0]:
data1=[('1','Kid1'),('2','Kid2')]
data2=[('Kid3','3'),('Kid4','4')]
schema1=('id String','name String')
schema2=('name String','id String')

df_1=spark.createDataFrame(data=data1,schema=schema)
df_2=spark.createDataFrame(data=data2,schema=schema)
df_3=df_1.union(df_2)
display(df_3)

## Using unionByName 

In [0]:
data1=[('1','Kid1'),('2','Kid2')]
data2=[('Kid3','3'),('Kid4','4')]
schema1='id STRING, name STRING'
schema2='name STRING, id STRING'

df_1=spark.createDataFrame(data=data1,schema=schema1)
df_2=spark.createDataFrame(data=data2,schema=schema2)
df_3=df_1.unionByName(df_2)
# display(df_1.columns)

display(df_3)

## String functions 
### INITCAP()
### UPPER()
### LOWER()

### INITCAP()

In [0]:
df.select(initcap('Item_Type').alias('Init_Item_Type')).display()

### UPPPER

In [0]:
df.select(upper('Item_Type').alias('Upper_Item_Type')).display()

### LOWER()

In [0]:
df.select(lower('Item_Type').alias('Lower_Item_Type')).display()

## Date functions
### CURRENT_DATE()
### DATE_ADD()
### DATE_SUB()

## CURRENT_DATE()

In [0]:
df=df.withColumn('curr_date',current_date())
df.display()

## DATE_ADD()

In [0]:
df=df.withColumn('week_after',date_add(col('curr_date'),7))
df.display()

## Date_Sub()

In [0]:
df.withColumn('week_before',date_sub('curr_date',7)).display()

### using date_add to get a week before column

In [0]:
df=df.withColumn('week_before',date_add('curr_date',-7))
df.display()

### Datediff


In [0]:
df=df.withColumn('date_diff',date_diff('week_after','week_before'))
df.display()                                    

### Dateformat - It can be used to change the format of the date column 

#### Transform the existing column using withcokumn and change the format 

In [0]:
df=df.withColumn('week_before',date_format('curr_date','dd-MM-yyyy'))
df.display()

### Handling NULLS

### Category 1- Dropping nulls
### Category 2-Filling nulls

### Dropping nulls

#### Dropna-two options any and all -any means drop the row if any column has null an all means only drop when all the columns has null in it.

In [0]:
df.dropna('all').display()

In [0]:
df.dropna('any').display()

In [0]:
df.createOrReplaceTempView("df")
spark.sql("SELECT * FROM df WHERE Item_Identifier = 'FDG12'").display()

### option 3 is to use subset where we can specify the records to be dropped based on which column.

In [0]:
df.dropna(subset=['Outlet_Size']).display()

### Scenario 2 -Fill the null values 

#### Again two options replace all the null values or replace the null values in specified columns only

In [0]:
df.fillna('Not Available').display()

In [0]:
df.fillna('NotAvailable',subset=['Outlet_Size']).display()

### Split and Indexing 
#### Split is used to divide our data based on space or comma it is used when we have a multivalued attribute and then indexing is used to pick the part we want and we can use the same to create new cols or anything.

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')[1]).display()

#### Explode -It is used when we have multiple values in a cell or column and we want to create a new record for each value in the column we can use explode after splitting the data 
#### Example -- H1 'abc def'
#### After split-- H1 ['abc','def']
#### After explode-H1 'abc'
####              -H1 'def'


In [0]:
df_exp=df.withColumn('Outlet_Type',split('Outlet_Type',' '))
display(df_exp)

In [0]:
df_exp.withColumn('Outlet_Type',explode('Outlet_Type')).display()

#### Array Contains--It is used to check if the text is present in the array list or not. It gives boolean as output.

In [0]:
df_exp.withColumn('Type_1_flag',array_contains('Outlet_Type','Type1')).display()

### Group by

#### Scenario 1- Total mrp of each item type

In [0]:
df.groupBy('Item_Type').agg(sum('Item_MRP').alias('Total_MRP')).display()

#### Scenario 2-Group by item type and then on outlet size and then find the total of Item_MRP

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Total_Mrp')).display()

#### Scenario 4-group by on two cols and agg on two cols

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Total_Mrp'),avg('Item_MRP').alias('Avg_Mrp')).display()

### Collect list -this is used when we want to group by some columns and instead of performing aggregations on it we want the data in form of array list. 

In [0]:
data=[('user1','book1')
      ,('user1','book2'),
      ('user2','book1'),
      ('user2','book2'),
      ('user3','book1'),
      ('user3','book2'),
      ('user3','book3'),
      ('user4','book1')]
schema='user String,book String'
df_user=spark.createDataFrame(data,schema)
df_user.groupBy('user').agg(collect_list('book').alias('books_per_user')).display()

### Pivot-

In [0]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).display()

#### When Otherwise - It is a function that is used to give conditional values to a column.

#### Scenario 1- We are using when otherwise function to derive a new column which is veg or non veg based on values present in the item_type column.

In [0]:
df.withColumn('veg_flag',when(col('Item_Type')=='Meat','Non_Veg').otherwise('Veg')).display()


#### Scenario 2- If Item_Type is veg and Item_MRP is >100 then it will be veg expensive else if it is veg then veg inexpensive else non veg

In [0]:
df.withColumn('flag',when(col('Item_Type')!='Meat',when(col('Item_MRP')>100,'Veg Expensive').otherwise('Veg Cheap')).otherwise('Non Veg')).display()

### Joins----
#### Inner 
#### Left
#### Right
#### Full 
#### Anti join

#### Inner join

In [0]:
emp=[(1,'Gaur','d01'),
     (2,'Ravi','d02'),
     (3,'Alice','d03'),
     (4,'Sam','d03'),
     (5,'John','d05')]
schema1='id int, name string, dept_id string'
emp_df=spark.createDataFrame(emp,schema1)
display(emp_df)

In [0]:
dept=[('d01','HR'),
     ('d02','IT'),
     ('d03','Sales'),
     ('d04','Marketing')]

dept_schema='dept_id string, dept_name string'
df_dept=spark.createDataFrame(dept,dept_schema)
display(df_dept)

#### Inner Join

In [0]:
emp_df.join(df_dept,emp_df['dept_id']==df_dept['dept_id'],'inner').select('id','name',emp_df['dept_id'],'dept_name').display()

#### Left join

In [0]:
emp_df.join(df_dept,emp_df['dept_id']==df_dept['dept_id'],'left').select('id','name',emp_df['dept_id'],'dept_name').display()


#### Right join

In [0]:
emp_df.join(df_dept,emp_df['dept_id']==df_dept['dept_id'],'right').select('id','name',df_dept['dept_id'],'dept_name').display()

#### Anti join- when we have to fetch the data which is available in one dataframe and it is not available in the other data frame.


##### Fetch the list of employees whose department is not present in the department table.

In [0]:
emp_df.join(df_dept,emp_df['dept_id']==df_dept['dept_id'],'left_anti').display()


### Window functions


#### Rownumber -It gives us the unique number to each record.It is used to remove the duplicates or generate surrogate key.

#### over is used to specify which column to use while generating the numbers

In [0]:
from pyspark.sql.window import Window

#### Row number

In [0]:
df.withColumn('row_num',row_number().over(Window.orderBy('Item_Identifier'))).display()

### Rank and dense rank
### Both are used to give ranking to the records based on any column,but the rank gives same numbers in case of tie and skips the duplicate ranks count and assign the next rank eg:-1,1,1,4
### Dense rank-1,1,1,2

In [0]:
df.withColumn('rank',rank().over(Window.orderBy('Item_Identifier'))).display()

In [0]:
df.withColumn('rank',rank().over(Window.orderBy(col('Item_Identifier').desc())))\
    .withColumn('denserank',dense_rank().over(Window.orderBy(col('Item_Identifier').desc()))).display()

#### Cumulative sum

#### In the below code it is giving us the total sum for each Item_Type but we want the cumulative sum.

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type'))).display()

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

#### User defined functions --We can create our own python functions and use it in spark if we think there is no function avaliable in spark to do it or it is very complex. But it is not recommended to use udf due to performance issues.

#### The issue is by default our executor uses jvm for execution but if you write any udf then it will need python interpretor and more resources as well to execute the code which will cause performance issues.

#### Step 1- Create a python function

In [0]:
def my_function(x):
    return x*x

### Step 2- Convert it into pyspark function

In [0]:
my_udf=udf(my_function)


In [0]:
df.withColumn('new_col',my_udf(col('Item_MRP'))).display()

#### Data writing(Serving layer)

#### Writing data in csv format.

In [0]:
df.write.mode('overwrite').saveAsTable('data_csv')

In [0]:
df2 = spark.read.table('workspace.default.data_csv')
display(df2)

#### Data writing modes in pyspark
#### Append
#### Override
#### Error 
#### Ignore


#### Append is used when we already have a file and we need to add more data in the file
#### Override is used when we want to replace the existing data and add new data.I is mostly used when we want to create stage layers.
#### Error mode will throw error if there is existing data at the destination.
#### Ignore-If there is already a file at the destination it will just ignore it and wont do anything.

#### Parquet file format- It is the first choice when we are working with big data as it is optimized.
There are two ways of storing the data row based is optimal for oltp databases as the data is stored in row format together but in column based storage the entire data of one column is stored together so it is optimal for reading 

In [0]:
df.write.format('parquet').mode('overwrite').saveAsTable('data_parquet')


There are two ways of storing the data 
1. Using the saveAsTable() method
2. Using the write() method with the format() method
df.write.mode('overwrite').format('parquet').saveAsTable('data_parquet')
df2 = spark.read.table('data_parquet')
display(df2)
df.write.mode('overwrite').format('parquet').save('/FileStore/tables/data_parquet')

Unity Catalog Tables (using .saveAsTable())
Unity Catalog managed tables only support Delta format. This is why your Parquet code failed. Delta is the recommended format for tables because it provides:

ACID transactions
Time travel
Schema enforcement
Better performance with optimizations
# Always creates a Delta table
df.write.mode('overwrite').saveAsTable('my_table')
2. File-Based Storage (using .save() with a path)
If you want to save data in Parquet, CSV, JSON, ORC, or Avro formats, write to a Unity Catalog Volume path instead:


# ORC format
df.write.format('orc').mode('overwrite').save('/Volumes/workspace/default/sales_data/my_data.orc')
Available Formats:
Delta - Optimized lakehouse format (default for tables)
Parquet - Columnar storage, great for analytics
CSV - Human-readable text format
JSON - Semi-structured data format
Avro - Row-based format with schema evolution
ORC - Optimized row columnar format
Best Practice:
For most use cases, use Delta tables (.saveAsTable()) because they provide the best performance and features. Only use file formats when you need to:

Share data with external systems that don't support Delta
Archive data in a specific format
Interoperate with non-Databricks tools

In [0]:
df.write.format('parquet').mode('overwrite').save('/Volumes/workspace/default/sales_data/my_data.parquet')

Unity Catalog Tables
What they are:

Structured datasets stored in Delta Lake format (or external formats for unmanaged tables)
Designed for analytical queries using SQL or PySpark DataFrames
Stored data with schema enforcement (defined columns and data types)
Key Features:

ACID transactions
Time travel and versioning
Schema evolution
Optimized for querying with Spark SQL
Automatic data optimization
Use Cases:

Storing structured business data (customer records, sales transactions, logs)
Building data pipelines (bronze, silver, gold layers)
Analytics and BI dashboards
Machine learning feature tables
Any data you need to query with SQL


Unity Catalog Volumes
What they are:

File storage locations for unstructured or semi-structured data
Similar to cloud storage (S3, ADLS) but managed by Unity Catalog
Store any file type (CSV, JSON, Parquet, images, videos, PDFs, models)
Key Features:

Governed file storage with Unity Catalog permissions
Can store files in their native format
Access via file paths (/Volumes/catalog/schema/volume/)
No schema enforcement
Support for non-tabular data
Use Cases:

Storing raw files before processing (CSV, JSON, XML uploads)
Machine learning artifacts (trained models, checkpoints)
Unstructured data (images, videos, PDFs, audio)
Application data (config files, scripts)
Staging area for data ingestion
Checkpoint directories for streaming jobs
Example:

Quick Comparison
Aspect
Tables
Volumes
Data Type
Structured/tabular
Unstructured/files
Format
Delta (managed tables)
Any format
Access
SQL, DataFrame APIs
File system paths
Schema
Enforced
None
Best For
Queries & analytics
File storage
Example Path
workspace.default.sales_data
/Volumes/workspace/default/sales_data/file.csv
Common Workflow Pattern
Upload raw files → Store in Volume (/Volumes/.../raw/)
Read and process → Load files into DataFrames
Save processed data → Write as Tables for querying
Query for insights → Use SQL on the tables
Example:



Delta format which is used to build db tables is built on top of parquet format.In parquet format the metadata is stored in the footer of the file
In delta format the metadata is used in transaction log or delta log which is used for schema enforcement and schema evolution.

## Managed vs External tables
### Managed table is the table which is created in databricks and stored in default databricks location and is managed by databricks if you delete the table data will also be deleted
### The data will be stored somewere else like on cloud or something so if you delete the table only schema will be deleted not the data.

In [0]:
display(df)

### To use sql we need to convert out df into temproary view and then use sql on top of it.

In [0]:
df.createTempView('my_view')

In [0]:
%sql
select * from my_view

If you want to convert sql statement result into a dataframe then you can use spark sql.

In [0]:
df_sql=spark.sql("Select * from my_view")
display(df_sql)